In [1]:
import sys
import torch
sys.path.append("var/cuda_gelu")
from inline_gelu import gelu as cuda_gelu  # type: ignore

In [2]:
import time
import torch
from collections.abc import Callable

def benchmark(description: str, run: Callable) -> None:
    nwarmups = 3
    nruns = 10
    for _ in range(nwarmups):
        run()
    torch.cuda.synchronize()
    durs = []
    for _ in range(nruns):
        t0 = time.time()
        run()
        torch.cuda.synchronize()
        durs.append(time.time() - t0)
    avg = sum(durs) / len(durs) * 1000
    print(f"{description}: {avg:.2f} ms")

def run_op1(dim: int, op: Callable) -> Callable:
    x = torch.randn(dim, device="cuda:0")
    return lambda: op(x)

def manual_gelu(x: torch.Tensor) -> torch.Tensor:
    return 0.5 * x * (1.0 + torch.tanh(0.79788456 * (x + 0.044715 * x.pow(3))))

torch_gelu = lambda x: torch.nn.functional.gelu(x, approximate="tanh")
compiled_gelu = torch.compile(manual_gelu)

In [3]:
import triton
import triton.language as tl

def triton_gelu(x: torch.Tensor):
    assert x.is_cuda
    assert x.is_contiguous()
    # Allocate output tensor
    y = torch.empty_like(x)
    # Determine grid (elements divided into blocks)
    num_elements = x.numel()
    block_size = 1024  # Number of threads
    num_blocks = triton.cdiv(num_elements, block_size)
    triton_gelu_kernel[(num_blocks,)](x, y, num_elements, BLOCK_SIZE=block_size)
    return y

@triton.jit
def triton_gelu_kernel(x_ptr, y_ptr, num_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    # Indices where this thread block should operate
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    # Handle boundary
    mask = offsets < num_elements
    # Read
    x = tl.load(x_ptr + offsets, mask=mask)
    # Approx gelu is 0.5 * x * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x^3)))
    # Compute (tl.tanh doesn't exist, use tanh(a) = (exp(2a) - 1) / (exp(2a) + 1)
    a = 0.79788456 * (x + 0.044715 * x * x * x)
    exp = tl.exp(2 * a)
    tanh = (exp - 1) / (exp + 1)
    y = 0.5 * x * (1 + tanh)
    # Store
    tl.store(y_ptr + offsets, y, mask=mask)

In [5]:
dim = 4096 ** 2
benchmark("cuda_gelu    ", run_op1(dim, cuda_gelu))
benchmark("manual_gelu  ", run_op1(dim, manual_gelu))
benchmark("torch_gelu   ", run_op1(dim, torch_gelu))
benchmark("compiled_gelu", run_op1(dim, compiled_gelu))
benchmark("triton_gelu  ", run_op1(dim, triton_gelu))

cuda_gelu    : 0.21 ms
manual_gelu  : 1.80 ms
torch_gelu   : 0.21 ms
compiled_gelu: 0.27 ms
triton_gelu  : 0.24 ms
